# || NEMO Benchmarking .tiff generator ||
© Konstantinos Andreadis 2024 (PhD in the Roux Lab & Salbreux Lab at UNIGE, Switzerland)

In [ ]:
%load_ext autoreload
%autoreload 2
# Import custom module_scripts
from module_scripts import datahandler, visuals

# Import custom module_scripts

import numpy as np
import tifffile
from scipy.ndimage import map_coordinates, gaussian_filter
import os

In [ ]:

def advect_fibers(seeds, mask, flow, z, y, x, n_steps, dt=0.8):
    """Helper to advect seeds along a vector flow field."""
    active_seeds = seeds * mask
    acc = np.copy(active_seeds)
    for d in [1, -1]:
        cz, cy, cx = z.astype(float), y.astype(float), x.astype(float)
        for _ in range(n_steps):
            cx += d * dt * flow[0]
            cy += d * dt * flow[1]
            cz += d * dt * flow[2]
            acc += map_coordinates(active_seeds, [cz, cy, cx], order=1, mode='constant', cval=0)
    res = np.zeros_like(acc)
    res[mask] = acc[mask]
    return res


def get_tetrahedral_flow(dx, dy, dz, pts):
    """Calculates the +1/2 defect tetrahedral 'tennis ball' flow for given local coordinates."""
    dist = np.sqrt(dx ** 2 + dy ** 2 + dz ** 2) + 1e-9
    nx, ny, nz = dx / dist, dy / dist, dz / dist

    # Standard spherical coordinates
    phi = np.arctan2(dy, dx)
    theta = np.arccos(np.clip(nz, -1, 1))

    et = [np.cos(phi) * np.cos(theta), np.sin(phi) * np.cos(theta), -np.sin(theta)]
    ep = [-np.sin(phi), np.cos(phi), np.zeros_like(phi)]

    # FIX: Explicitly set dtype to float to avoid UFuncTypeError during +=
    v_sum_x = np.zeros_like(dx, dtype=np.float32)
    v_sum_y = np.zeros_like(dx, dtype=np.float32)

    for p in pts:
        v_vec = [nx - p[0], ny - p[1], nz - p[2]]
        v_t = v_vec[0] * et[0] + v_vec[1] * et[1] + v_vec[2] * et[2]
        v_p = v_vec[0] * ep[0] + v_vec[1] * ep[1] + v_vec[2] * ep[2]
        v_mag = np.sqrt(v_t ** 2 + v_p ** 2 + 1e-9)

        v_sum_x += v_t / v_mag
        v_sum_y += v_p / v_mag

    inner_angle = 0.5 * np.arctan2(v_sum_y, v_sum_x)
    flow = [
        et[0] * np.cos(inner_angle) + ep[0] * np.sin(inner_angle),
        et[1] * np.cos(inner_angle) + ep[1] * np.sin(inner_angle),
        et[2] * np.cos(inner_angle) + ep[2] * np.sin(inner_angle)
    ]
    return flow, dist


def generate_shifted_shells(l=256, r_inner=80, r_outer=110, dr=1.5, steps=30, seed_density=0.012):
    """
    Showcase 1: Shifted, Rotated Double Shell.
    Both shells have half-defects, but the outer shell is shifted off-center
    and the defect orientation is rotated. The space between is filled with low intensity.
    """
    print(">> Initializing Shifted Double-Shell Simulation...")
    vol = np.zeros((l, l, l), dtype=np.float32)
    z, y, x = np.indices((l, l, l))
    seeds = (np.random.uniform(0, 1, (l, l, l)) > (1 - seed_density)).astype(np.float32)

    # Base tetrahedral points
    s = 1 / np.sqrt(3)
    pts_base = [
        np.array([s, s, s]),
        np.array([s, -s, -s]),
        np.array([-s, s, -s]),
        np.array([-s, -s, s])
    ]

    # --- INNER SHELL ---
    print(">> Calculating Centered Inner Shell...")
    center_in = np.array([l // 2, l // 2, l // 2])
    dx_in, dy_in, dz_in = x - center_in[0], y - center_in[1], z - center_in[2]

    flow_in, dist_in = get_tetrahedral_flow(dx_in, dy_in, dz_in, pts_base)
    mask_in = (dist_in >= r_inner - dr) & (dist_in <= r_inner + dr)

    vol += advect_fibers(seeds, mask_in, flow_in, z, y, x, steps) * 180
    vol[mask_in] += 40

    # --- OUTER SHELL ---
    print(">> Calculating Shifted & Rotated Outer Shell...")
    # Shift center by 8 pixels in X and 5 in Y
    center_out = center_in + np.array([8.0, 5.0, 0.0])
    dx_out, dy_out, dz_out = x - center_out[0], y - center_out[1], z - center_out[2]

    # Rotate the outer defects by 45 degrees (pi/4) around Z axis
    theta_rot = np.pi / 4
    Rz = np.array([
        [np.cos(theta_rot), -np.sin(theta_rot), 0],
        [np.sin(theta_rot), np.cos(theta_rot), 0],
        [0, 0, 1]
    ])
    pts_rot = [Rz @ p for p in pts_base]

    flow_out, dist_out = get_tetrahedral_flow(dx_out, dy_out, dz_out, pts_rot)
    mask_out = (dist_out >= r_outer - dr) & (dist_out <= r_outer + dr)

    vol += advect_fibers(seeds, mask_out, flow_out, z, y, x, steps) * 180
    vol[mask_out] += 40

    # --- GAP FILL ---
    print(">> Filling inter-shell space with low intensity...")
    gap_mask = (dist_in > r_inner + dr) & (dist_out < r_outer - dr)
    vol[gap_mask] = 25  # Low uniform intensity

    vol = gaussian_filter(vol, sigma=0.5)
    print(">> Shifted Shells Benchmark saved successfully.")
    return vol


def generate_diagonal_cylinder(l=256, r_cyl=70, dr=1.5, steps=30, seed_density=0.012, alpha_deg=45):
    """
    Showcase 2: Diagonal Cylinder.
    Perfect geometry for testing Curvature (K=0, H=1/2R) and Nematic tensor lift logic.
    """
    print(f">> Initializing Diagonal Cylinder Simulation (Alpha = {alpha_deg} deg)...")
    vol = np.zeros((l, l, l), dtype=np.float32)
    center = np.array([l // 2, l // 2, l // 2])
    z, y, x = np.indices((l, l, l))
    seeds = (np.random.uniform(0, 1, (l, l, l)) > (1 - seed_density)).astype(np.float32)

    # Cylinder coordinates (oriented along Z)
    dx = (x - center[0]).astype(np.float32)
    dy = (y - center[1]).astype(np.float32)
    dist_cyl = np.sqrt(dx ** 2 + dy ** 2) + 1e-9
    phi = np.arctan2(dy, dx)

    # Tangential basis for a cylinder
    e_theta = [-np.sin(phi), np.cos(phi), np.zeros_like(phi)]
    e_z = [np.zeros_like(phi), np.zeros_like(phi), np.ones_like(phi)]

    # Diagonal flow
    alpha = np.radians(alpha_deg)
    flow_cyl = [
        e_theta[0] * np.cos(alpha) + e_z[0] * np.sin(alpha),
        e_theta[1] * np.cos(alpha) + e_z[1] * np.sin(alpha),
        e_theta[2] * np.cos(alpha) + e_z[2] * np.sin(alpha)
    ]

    # Mask cylinder body, capping the ends slightly before the image boundary
    mask_cyl = (dist_cyl >= r_cyl - dr) & (dist_cyl <= r_cyl + dr)
    z_mask = (z > 20) & (z < l - 20)
    final_mask = mask_cyl & z_mask

    vol += advect_fibers(seeds, final_mask, flow_cyl, z, y, x, steps) * 180
    vol[final_mask] += 40

    vol = gaussian_filter(vol, sigma=0.5)
    print(">> Cylinder Benchmark saved successfully.")
    return vol


# --- NEW SHOWCASE: CYTOKINETIC DUMBBELL ---

def generate_cytokinetic_dumbbell(l=256, r_lobe=60, lobe_sep=50, dr=1.5, steps=40, seed_density=0.015):
    """
    Showcase 3: Two-Sphere Fused Dumbbell.
    North Pole: +1 Aster
    Hinge (Neck): Azimuthal fibers (circumferential)
    South Pole: +1 Aster
    """
    print(">> Initializing Fused-Sphere Dumbbell Simulation...")
    vol = np.zeros((l, l, l), dtype=np.float32)
    center = np.array([l // 2, l // 2, l // 2])
    z, y, x = np.indices((l, l, l))
    seeds = (np.random.uniform(0, 1, (l, l, l)) > (1 - seed_density)).astype(np.float32)

    # Local coordinates relative to center
    dz = (z - center[0]).astype(np.float32)
    dy = (y - center[1]).astype(np.float32)
    dx = (x - center[2]).astype(np.float32)
    phi = np.arctan2(dy, dx)

    # 1. Define the Fused Geometry
    # We use the distance to two centers and take the smooth minimum
    dist_top = np.sqrt(dx ** 2 + dy ** 2 + (dz - lobe_sep) ** 2)
    dist_bot = np.sqrt(dx ** 2 + dy ** 2 + (dz + lobe_sep) ** 2)

    # Surface mask: where either sphere surface exists
    # Use a slightly soft union for the hinge
    surface_dist = np.minimum(dist_top, dist_bot)
    mask = (surface_dist >= r_lobe - dr) & (surface_dist <= r_lobe + dr)

    # 2. Define the Three Flow Regimes
    # A) Aster Flow (Radial from the respective centers)
    flow_aster_top = [dx, dy, dz - lobe_sep]
    flow_aster_bot = [dx, dy, dz + lobe_sep]

    # B) Azimuthal Flow (The Loop at the hinge)
    flow_loop = [-np.sin(phi), np.cos(phi), np.zeros_like(phi)]

    # 3. Blending Logic
    # Transition weights based on Z
    # weight_loop is 1 at dz=0 (hinge) and drops to 0 at the poles
    weight_loop = np.exp(-(dz / (lobe_sep * 0.7)) ** 4)

    # Select which aster to blend with based on hemisphere
    is_north = (dz > 0).astype(np.float32)

    # Initialize flow components as float32
    f_x = np.zeros_like(dx, dtype=np.float32)
    f_y = np.zeros_like(dx, dtype=np.float32)
    f_z = np.zeros_like(dx, dtype=np.float32)

    # Combine: (Aster * (1-weight)) + (Loop * weight)
    for i, (comp, a_top, a_bot, loop) in enumerate(zip([f_x, f_y, f_z],
                                                       flow_aster_top,
                                                       flow_aster_bot,
                                                       flow_loop)):
        aster_blend = a_top * is_north + a_bot * (1 - is_north)
        comp[:] = aster_blend * (1 - weight_loop) + loop * weight_loop

    # Normalize the final vector field
    f_mag = np.sqrt(f_x ** 2 + f_y ** 2 + f_z ** 2) + 1e-9
    flow_final = [f_x / f_mag, f_y / f_mag, f_z / f_mag]

    print(">> Advecting fibers along the transition...")
    vol += advect_fibers(seeds, mask, flow_final, z, y, x, steps) * 180
    vol[mask] += 40

    vol = gaussian_filter(vol, sigma=0.5)
    print(">> Dumbbell Benchmark completed.")
    return vol

In [ ]:
output_dir = '/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/!MANUSCRIPT_NEMO/simulated'
L = 256
# # Generate Showcase 1
# stack_shells = generate_shifted_shells(l=L, r_inner=80, r_outer=110)
# tifffile.imwrite(os.path.join(output_dir, "shells_shifted.tif"),
#                  stack_shells.astype(np.uint16))
# # Generate Showcase 2
# stack_cylinder = generate_diagonal_cylinder(l=L, r_cyl=80, alpha_deg=45)
# tifffile.imwrite(os.path.join(output_dir, "cylinder_diagonal.tif"),
#                  stack_cylinder.astype(np.uint16))

stack_dumbbell = generate_cytokinetic_dumbbell(l=L)
tifffile.imwrite(os.path.join(output_dir, 'dumbbell_bridge.tif'), stack_dumbbell.astype(np.uint16))

In [ ]:
visuals.plot_img(img=stack_shells, scale=(1, 1, 1), unit="px", max_proj=False)
visuals.plot_img(img=stack_shells, scale=(1, 1, 1), unit="px", max_proj=True)
visuals.plot_img(img=stack_cylinder, scale=(1, 1, 1), unit="px", max_proj=False)
visuals.plot_img(img=stack_cylinder, scale=(1, 1, 1), unit="px", max_proj=True)
visuals.plot_img(img=stack_dumbbell, scale=(1, 1, 1), unit="px", max_proj=False)
visuals.plot_img(img=stack_dumbbell, scale=(1, 1, 1), unit="px", max_proj=True)

In [ ]:
visuals.view_img([stack_shells, stack_cylinder, stack_dumbbell],
                 scale=(1, 1, 1), color_list=["green", "red", "blue"])

# 3D Hedgehog

In [ ]:
def fibonacci_sphere(n_points):
    """Return approximately uniform directions over a sphere."""
    i = np.arange(n_points)
    phi = np.arccos(1 - 2 * (i + 0.5) / n_points)  # polar angle
    theta = np.pi * (1 + 5 ** 0.5) * (i + 0.5)  # golden angle (azimuth)
    return phi, theta


n_rays = 100  # Adjust for density
Nz, Ny, Nx = 200, 200, 200
scale = (1, 1, 1)
test = np.zeros((Nz, Ny, Nx))

# Center of the star
center = np.array([Nz // 2, Ny // 2, Nx // 2])

# Get uniformly distributed directions
phi, theta = fibonacci_sphere(n_rays)

# Ray directions
num_steps = 300
r = np.linspace(0, min(Nz, Ny, Nx), num_steps)

dz = np.outer(np.cos(phi), r)
dy = np.outer(np.sin(phi) * np.sin(theta), r)
dx = np.outer(np.sin(phi) * np.cos(theta), r)

# Add to center
z = dz + center[0]
y = dy + center[1]
x = dx + center[2]

# Round and index
z_idx = np.round(z).astype(int).ravel()
y_idx = np.round(y).astype(int).ravel()
x_idx = np.round(x).astype(int).ravel()

# Mask in-bounds
valid = (
        (z_idx >= 0) & (z_idx < Nz) &
        (y_idx >= 0) & (y_idx < Ny) &
        (x_idx >= 0) & (x_idx < Nx)
)

# Use boolean mask to prevent accumulation
mask = np.zeros_like(test, dtype=bool)
mask[z_idx[valid], y_idx[valid], x_idx[valid]] = True
test[mask] = 1
datahandler.save_tiff(test, "/Users/andreadi/Downloads/3dstar.tiff", img_unit="um")
# visuals.view_img([test], scale=(1, 1, 1))